In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/arjunmahesh09999/new-set02-0/New_set.csv")

print("Unique patients:", df["patient_id"].nunique())

Unique patients: 103


In [5]:
import pandas as pd


#Rename dictionary
rename_dict = {
    "Solar8000/RR": "resp_rate",
    "Solar8000/PLETH_SPO2": "spo2",
    "Solar8000/HR": "heart_rate",
    "Solar8000/ART_DBP": "dbp",
    "Solar8000/ART_MBP": "mbp",
    "Solar8000/ART_SBP": "sbp",
    "Solar8000/ETCO2": "etco2"
}

# Rename columns
df = df.rename(columns=rename_dict)

# Save cleaned version
df.to_csv("renamed_patients.csv", index=False)

print("Renaming completed.")
print("New columns:")
print(df.columns)

Renaming completed.
New columns:
Index(['patient_id', 'time', 'dbp', 'mbp', 'heart_rate', 'resp_rate', 'sbp',
       'spo2', 'etco2', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11',
       'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15',
       'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18'],
      dtype='object')


In [6]:
df =df.sort_values(['patient_id', 'time'])


In [7]:
print(df.head())

        patient_id  time   dbp   mbp  heart_rate  resp_rate   sbp  spo2  \
488516          64   0.0   NaN   NaN         NaN        NaN   NaN   NaN   
488517          64   2.0   NaN   NaN         NaN        NaN   NaN   NaN   
488518          64   4.0 -20.0 -20.0        51.0       11.0 -20.0  97.0   
488519          64   6.0 -20.0 -20.0        51.0       10.0 -20.0  98.0   
488520          64   8.0 -20.0 -20.0        51.0       10.0 -20.0  98.0   

        etco2  Unnamed: 9  Unnamed: 10  Unnamed: 11  Unnamed: 12  Unnamed: 13  \
488516    NaN         NaN          NaN          NaN          NaN          NaN   
488517    NaN         NaN          NaN          NaN          NaN          NaN   
488518    NaN         NaN          NaN          NaN          NaN          NaN   
488519    NaN         NaN          NaN          NaN          NaN          NaN   
488520    NaN         NaN          NaN          NaN          NaN          NaN   

        Unnamed: 14  Unnamed: 15  Unnamed: 16  Unnamed: 17  Un

In [8]:



# Count NaN per column
nan_counts = df.isna().sum()

# Show only columns that actually have NaN
nan_counts = nan_counts[nan_counts > 0]

print("Columns with NaN values:")
print(nan_counts)

Columns with NaN values:
time               62
dbp             40691
mbp             27600
heart_rate      34651
resp_rate       33533
sbp             40671
spo2            27661
etco2           33894
Unnamed: 9     713019
Unnamed: 10    713019
Unnamed: 11    713019
Unnamed: 12    713019
Unnamed: 13    713019
Unnamed: 14    713019
Unnamed: 15    713019
Unnamed: 16    713019
Unnamed: 17    713019
Unnamed: 18    713019
dtype: int64


In [9]:


# remove unnamed columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# remove rows where time is NaN
df = df.dropna(subset=["time"])

print(df.shape)

(712957, 9)


In [10]:
df["pulse_pressure"] = df["sbp"] - df["dbp"]

In [11]:
vital_cols = [
    "spo2","heart_rate","resp_rate",
    "sbp","dbp","mbp",
    "etco2","pulse_pressure"
]

df = df.sort_values(["patient_id","time"]).reset_index(drop=True)

df[vital_cols] = (
    df.groupby("patient_id")[vital_cols]
      .transform(lambda x: x.ffill().bfill())
)

In [12]:
# Count NaN per column
nan_counts = df.isna().sum()

# Show only columns that actually have NaN
nan_counts = nan_counts[nan_counts > 0]

print("Columns with NaN values:")
print(nan_counts)

Columns with NaN values:
Series([], dtype: int64)


In [13]:
df.head()

,patient_id,time,dbp,mbp,heart_rate,resp_rate,sbp,spo2,etco2,pulse_pressure
0,64,0.0,-20.0,-20.0,51.0,11.0,-20.0,97.0,1.0,0.0
1,64,2.0,-20.0,-20.0,51.0,11.0,-20.0,97.0,1.0,0.0
2,64,4.0,-20.0,-20.0,51.0,11.0,-20.0,97.0,1.0,0.0
3,64,6.0,-20.0,-20.0,51.0,10.0,-20.0,98.0,1.0,0.0
4,64,8.0,-20.0,-20.0,51.0,10.0,-20.0,98.0,1.0,0.0


In [14]:
output_path = "/kaggle/working/newbefore_clean.csv"
df.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: /kaggle/working/newbefore_clean.csv


In [1]:
import sys
import pandas as pd
import numpy as np

# =======================
# CONFIG / CONSTANTS
# =======================
CRITICAL_THRESHOLD  = 0.75
EMERGENCY_THRESHOLD = 1.5

BASE_1_2 = 1.4672
BASE_3   = 1.19
EXTRA_T1, EXTRA_T2, EXTRA_T3 = 1.30, 1.20, 1.10
MULTIPLIER_CAP = 2.2

MIN_COND_ACTIVATION = 0.20
EARLY_FRAC = 0.20   # ramp starts 20% before the condition-specific physiological threshold

RR_SMOOTH_WINDOW = 5

# Slope window sizes in rows (assumes 2-second row interval)
#   2 min  =  60 rows
#   5 min  = 150 rows
#   7 min  = 210 rows
#  15 min  = 450 rows
SLOPE_WINDOWS = {"2m": 60, "5m": 150, "7m": 210, "15m": 450}

# Rolling window sizes (same row counts as slope windows)
ROLL_WINDOWS = {"2m": 60, "5m": 150, "7m": 210, "15m": 450}

# Lag window (15 minutes = 450 rows)
LAG_15M = 450

# Raw vital columns to compute slopes for (not z_ or s_)
RAW_SLOPE_VITALS = [
    "spo2", "heart_rate", "resp_rate_smoothed",
    "sbp", "dbp", "mbp", "etco2", "pulse_pressure",
]

# Vitals to generate lag_15m features for
LAG_VITALS = [
    "spo2", "heart_rate", "resp_rate_smoothed",
    "sbp", "dbp", "mbp", "etco2", "pulse_pressure",
]

# =======================
# THRESHOLDS (N, C, E)
# =======================
TH = {
    "spo2":      (95,  92,  90),
    "hr_low":    (60,  50,  45),   "hr_high":   (90,  110, 120),
    "rr_low":    (12,  10,   8),   "rr_high":   (20,   25,  30),
    "sbp_low":   (110, 100,  90),  "sbp_high":  (150, 170, 185),
    "dbp_low":   (60,   55,  50),  "dbp_high":  (85,   95, 100),
    "mbp":       (70,   65,  60),
    "etco2_low": (35,   30,  25),  "etco2_high":(45,   50,  55),
    "pp_low":    (45,   35,  30),  "pp_high":   (65,   75,  85),
}

# =======================
# Per-condition factor specs
# =======================
CONDITION_FACTOR_SPECS = {
    # ---- TIER 1 ----
    "t1_shock_spiral": [
        {"raw": "mbp",                "dir": "low",  "thresh": 70,  "normal_ref": 90},
        {"raw": "heart_rate",         "dir": "high", "thresh": 100, "normal_ref": 75},
    ],
    "t1_resp_burnout": [
        {"raw": "spo2",               "dir": "low",  "thresh": 92,  "normal_ref": 98},
        {"raw": "resp_rate_smoothed", "dir": "high", "thresh": 22,  "normal_ref": 16},
    ],
    "t1_hypercapnic": [
        {"raw": "etco2",              "dir": "high", "thresh": 50,  "normal_ref": 40},
        {"raw": "resp_rate_smoothed", "dir": "low",  "thresh": 10,  "normal_ref": 16},
    ],

    # ---- TIER 2 ----
    "t2_pulse_pressure_low": [
        {"raw": "pulse_pressure",     "dir": "low",  "thresh": 30,  "normal_ref": 50},
    ],
    "t2_widepp_highsbp": [
        {"raw": "pulse_pressure",     "dir": "high", "thresh": 70,  "normal_ref": 50},
        {"raw": "sbp",                "dir": "high", "thresh": 170, "normal_ref": 120},
    ],
    "t2_resp_hemo_combo": [
        {"raw": "spo2",               "dir": "low",  "thresh": 92,  "normal_ref": 98},
        {"raw": "resp_rate_smoothed", "dir": "high", "thresh": 22,  "normal_ref": 16},
        {"raw": "heart_rate",         "dir": "high", "thresh": 100, "normal_ref": 75},
    ],

    # ---- TIER 3 ----
    "t3_hyper_emergency": [
        {"raw": "sbp",                "dir": "high", "thresh": 180, "normal_ref": 120},
        {"raw": "pulse_pressure",     "dir": "high", "thresh": 70,  "normal_ref": 50},
    ],
    "t3_stable_deceiver": [
        {"raw": "spo2",               "dir": "low",  "thresh": 94,  "normal_ref": 98},
        {"raw": "heart_rate",         "dir": "low",  "thresh": 90,  "normal_ref": 100},
        {"raw": "mbp",                "dir": "low",  "thresh": 70,  "normal_ref": 90},
    ],
    "t3_masked_shock": [
        {"raw": "mbp",                "dir": "low",  "thresh": 72,  "normal_ref": 90},
        {"raw": "heart_rate",         "dir": "low",  "thresh": 90,  "normal_ref": 100},
    ],
    "t3_occult_acidosis": [
        {"raw": "etco2",              "dir": "low",  "thresh": 32,  "normal_ref": 40},
        {"raw": "resp_rate_smoothed", "dir": "high", "thresh": 24,  "normal_ref": 16},
        {"raw": "spo2",               "dir": "low",  "thresh": 92,  "normal_ref": 98},
    ],
    "t3_trend_decline": [
        {"raw": "etco2",              "dir": "high", "thresh": 50,  "normal_ref": 40},
        {"raw": "spo2",               "dir": "low",  "thresh": 92,  "normal_ref": 98},
        {"raw": "heart_rate",         "dir": "high", "thresh": 100, "normal_ref": 75},
    ],
    "t3_trend_activate": [
        {"raw": "spo2",               "dir": "low",  "thresh": 94,  "normal_ref": 98},
        {"raw": "heart_rate",         "dir": "high", "thresh": 95,  "normal_ref": 75},
        {"raw": "etco2",              "dir": "low",  "thresh": 35,  "normal_ref": 40},
    ],
}

# Tier + activation details for every condition
CONDITIONS = {
    "t1_shock_spiral":       {"tier": 1},
    "t1_resp_burnout":       {"tier": 1},
    "t1_hypercapnic":        {"tier": 1},
    "t2_pulse_pressure_low": {"tier": 2},
    "t2_widepp_highsbp":     {"tier": 2},
    "t2_resp_hemo_combo":    {"tier": 2},
    "t3_hyper_emergency":    {"tier": 3},
    "t3_stable_deceiver":    {"tier": 3},
    "t3_masked_shock":       {"tier": 3},
    "t3_occult_acidosis":    {"tier": 3},
    "t3_trend_decline":      {"tier": 3},
    "t3_trend_activate":     {"tier": 3},
}

# =======================
# Z / severity helpers
# =======================
def z_low(x, N, C, E):
    if pd.isna(x): return 0.0
    if x >= N:     return 0.0
    if x >  C:     denom = N - C;  return (0.5 * (N - x) / denom) if denom != 0 else 0.5
    if x >  E:     denom = C - E;  return (0.5 + 0.5 * (C - x) / denom) if denom != 0 else 1.0
    return 1.0

def z_high(x, N, C, E):
    if pd.isna(x): return 0.0
    if x <= N:     return 0.0
    if x <  C:     denom = C - N;  return (0.5 * (x - N) / denom) if denom != 0 else 0.5
    if x <  E:     denom = E - C;  return (0.5 + 0.5 * (x - C) / denom) if denom != 0 else 1.0
    return 1.0

def severity_from_z(z): return (2.0 ** z) - 1.0

def z_spo2(v):  return z_low(v, *TH["spo2"])
def z_hr(v):    return max(z_low(v, *TH["hr_low"]),    z_high(v, *TH["hr_high"]))
def z_rr(v):    return max(z_low(v, *TH["rr_low"]),    z_high(v, *TH["rr_high"]))
def z_sbp(v):   return max(z_low(v, *TH["sbp_low"]),   z_high(v, *TH["sbp_high"]))
def z_dbp(v):   return max(z_low(v, *TH["dbp_low"]),   z_high(v, *TH["dbp_high"]))
def z_mbp(v):   return z_low(v, *TH["mbp"])
def z_etco2(v): return max(z_low(v, *TH["etco2_low"]), z_high(v, *TH["etco2_high"]))
def z_pp(v):    return max(z_low(v, *TH["pp_low"]),    z_high(v, *TH["pp_high"]))

# =======================
# Condition multipliers
# =======================
def base_for_tier(tier):
    return BASE_1_2 if tier in (1, 2) else BASE_3

def pair_target_for_tiers(tier_a, tier_b):
    tset = tuple(sorted([tier_a, tier_b]))
    if tset == (1, 1): return BASE_1_2 * EXTRA_T1
    if tset == (1, 2): return BASE_1_2 * EXTRA_T2
    if tset == (1, 3): return BASE_1_2 * EXTRA_T3
    if tset == (2, 2): return BASE_1_2 * EXTRA_T1
    if tset == (2, 3): return BASE_1_2 * EXTRA_T3
    if tset == (3, 3): return BASE_3  * EXTRA_T3
    return 1.0

# =======================
# Single-factor directional ramp (no pull)
# =======================
def compute_factor_A(raw_value, direction, thresh, normal_ref):
    if pd.isna(raw_value):
        return 0.0
    span = abs(thresh - normal_ref)
    if span < 1e-8:
        return 1.0 if (
            (direction == "high" and raw_value >= thresh) or
            (direction == "low"  and raw_value <= thresh)
        ) else 0.0

    buffer = EARLY_FRAC * span

    if direction == "high":
        early_start = thresh - buffer
        if raw_value <= early_start: return 0.0
        if raw_value >= thresh:      return 1.0
        denom = thresh - early_start
        return (raw_value - early_start) / denom if denom > 1e-8 else 1.0
    else:  # "low"
        early_start = thresh + buffer
        if raw_value >= early_start: return 0.0
        if raw_value <= thresh:      return 1.0
        denom = early_start - thresh
        return (early_start - raw_value) / denom if denom > 1e-8 else 1.0


# =======================
# Condition-level activation (plain mean of factor activations)
# =======================
def compute_condition_A(row, cond_name):
    specs = CONDITION_FACTOR_SPECS[cond_name]
    As = [
        compute_factor_A(
            float(row.get(s["raw"], np.nan)),
            s["dir"], s["thresh"], s["normal_ref"]
        )
        for s in specs
    ]
    if not any(a > 0.0 for a in As):
        return 0.0
    return float(np.mean(As))


# =======================
# OLS slope helper
# =======================
def _ols_slope(y_arr):
    """OLS slope of y over equally-spaced x=[0,1,...,n-1]. NaN-safe."""
    mask = ~np.isnan(y_arr)
    n_valid = mask.sum()
    if n_valid < 2:
        return np.nan
    x = np.where(mask)[0].astype(float)
    y = y_arr[mask]
    xm = x.mean(); ym = y.mean()
    denom = ((x - xm) ** 2).sum()
    if denom < 1e-10:
        return 0.0
    return float(((x - xm) * (y - ym)).sum() / denom)


def add_slopes(df, col, patient_col=None):
    """
    Add slope columns for `col` at all SLOPE_WINDOWS.
    Slopes are computed patient-wise if patient_col is given.
    Each row uses all available rows up to window_size (min_periods=2).
    Column names: slope_{win_name}_{col}
    """
    for win_name, win_size in SLOPE_WINDOWS.items():
        out_col = f"slope_{win_name}_{col}"
        if patient_col and patient_col in df.columns:
            df[out_col] = df.groupby(patient_col)[col].transform(
                lambda x: x.rolling(window=win_size, min_periods=2)
                           .apply(_ols_slope, raw=True)
            )
        else:
            df[out_col] = df[col].rolling(window=win_size, min_periods=2).apply(_ols_slope, raw=True)
    return df


# =======================
# Rolling stats on combined_score (mean, std, min, max)
# NaN at patient start (no reference point) → filled with 0
# =======================
def add_rolling_stats_combined(df, patient_col=None):
    """
    Adds rolling mean/std for all ROLL_WINDOWS and rolling min/max for 15m
    on combined_score, grouped by patient if patient_col is given.

    NaN produced when there is NO prior reference point (start of patient,
    fewer than min_periods rows available) are filled with 0.
    Downstream NaN (mid-patient gaps) are handled by existing forward-fill
    logic and are not touched here.
    """
    grp = df.groupby(patient_col)["combined_score"] if patient_col else df["combined_score"]

    for win_name, win_size in ROLL_WINDOWS.items():
        # ── rolling mean ──────────────────────────────────────────────
        col_mean = f"roll_mean_{win_name}_combined"
        if patient_col:
            df[col_mean] = grp.transform(
                lambda s: s.rolling(window=win_size, min_periods=2).mean()
            )
        else:
            df[col_mean] = df["combined_score"].rolling(window=win_size, min_periods=2).mean()
        # First-row (no reference) NaN → 0
        df[col_mean] = df[col_mean].fillna(0)

        # ── rolling std ───────────────────────────────────────────────
        col_std = f"roll_std_{win_name}_combined"
        if patient_col:
            df[col_std] = grp.transform(
                lambda s: s.rolling(window=win_size, min_periods=2).std()
            )
        else:
            df[col_std] = df["combined_score"].rolling(window=win_size, min_periods=2).std()
        # First-row (no reference) NaN → 0
        df[col_std] = df[col_std].fillna(0)

    # ── rolling min / max (15 min only, min_periods=1) ────────────────
    if patient_col:
        df["roll_min_15m_combined"] = grp.transform(
            lambda s: s.rolling(window=ROLL_WINDOWS["15m"], min_periods=1).min()
        )
        df["roll_max_15m_combined"] = grp.transform(
            lambda s: s.rolling(window=ROLL_WINDOWS["15m"], min_periods=1).max()
        )
    else:
        df["roll_min_15m_combined"] = (
            df["combined_score"].rolling(window=ROLL_WINDOWS["15m"], min_periods=1).min()
        )
        df["roll_max_15m_combined"] = (
            df["combined_score"].rolling(window=ROLL_WINDOWS["15m"], min_periods=1).max()
        )
    # First-row (no reference) NaN → 0
    df["roll_min_15m_combined"] = df["roll_min_15m_combined"].fillna(0)
    df["roll_max_15m_combined"] = df["roll_max_15m_combined"].fillna(0)

    return df


# =======================
# Lag-15m features for raw vitals + combined_score
# shift(LAG_15M) per patient; NaN at patient start (no reference) → 0
# =======================
def add_lag_features(df, patient_col=None):
    """
    For each vital in LAG_VITALS and for combined_score, adds a lag column:
        lag_15m_{col} = value from LAG_15M rows ago (within the same patient).

    Rows where NO prior reference exists (start of each patient, first
    LAG_15M rows) are filled with 0, NOT the patient's first value.
    """
    lag_targets = LAG_VITALS + ["combined_score"]

    for col in lag_targets:
        if col not in df.columns:
            continue

        lag_col = f"lag_15m_{col}"

        if patient_col and patient_col in df.columns:
            df[lag_col] = df.groupby(patient_col)[col].transform(
                lambda s: s.shift(LAG_15M)
            )
        else:
            df[lag_col] = df[col].shift(LAG_15M)

        # Only rows with NO reference point (truly unreferenced head of each
        # patient group) are filled with 0. Any mid-series NaN from gaps in
        # the original vital is already handled upstream.
        df[lag_col] = df[lag_col].fillna(0)

    return df


# =======================
# Tier / condition detection — returns binary series for EVERY condition
# Must be called AFTER raw vital slopes are added to df.
# =======================
def detect_conditions(df):
    """
    Returns a dict {condition_name: bool_series} for all conditions.
    Binary presence/absence based on physiological threshold logic.
    t3_trend_activate uses slope columns (must be pre-computed).
    """

    # ---- TIER 1 ----
    t1_shock_spiral = (df["mbp"] <= 70) & (df["heart_rate"] >= 100)

    t1_resp_burnout  = (df["spo2"] <= 92) & (df["resp_rate_smoothed"] >= 22)

    t1_hypercapnic   = (df["etco2"] >= 50) & (df["resp_rate_smoothed"] <= 10)

    # ---- TIER 2 ----
    t2_pulse_pressure_low = (df["pulse_pressure"] <= 30)

    t2_widepp_highsbp = (df["pulse_pressure"] >= 70) & (df["sbp"] >= 170)

    t2_resp_hemo_combo = (
        (df["spo2"] <= 92) &
        (df["resp_rate_smoothed"] >= 22) &
        (df["heart_rate"] >= 100)
    )

    # ---- TIER 3 ----
    t3_hyper_emergency = (df["sbp"] >= 180) & (df["pulse_pressure"] >= 70)

    t3_stable_deceiver = (
        df["spo2"].between(92, 94) &
        df["heart_rate"].between(75, 90) &
        df["mbp"].between(65, 70)
    )

    t3_masked_shock = (
        df["mbp"].between(65, 72) &
        (df["heart_rate"] < 90)
    )

    t3_occult_acidosis = (
        (df["etco2"] <= 32) &
        (df["resp_rate_smoothed"] >= 24) &
        df["spo2"].between(88, 92)
    )

    # Trend decline: point-to-point adverse change (uses prev columns)
    t3_trend_decline = (
        (df["etco2"]      - df["etco2_prev"]      > 0) &
        (df["spo2_prev"]  - df["spo2"]             > 0) &
        (df["heart_rate"] - df["heart_rate_prev"]  > 0)
    )

    # Trend activate: slope-based sustained deterioration
    sl_spo2_5m  = df.get("slope_5m_spo2",      pd.Series(np.nan, index=df.index))
    sl_hr_5m    = df.get("slope_5m_heart_rate", pd.Series(np.nan, index=df.index))
    sl_etco2_5m = df.get("slope_5m_etco2",      pd.Series(np.nan, index=df.index))
    sl_spo2_7m  = df.get("slope_7m_spo2",       pd.Series(np.nan, index=df.index))
    sl_hr_7m    = df.get("slope_7m_heart_rate",  pd.Series(np.nan, index=df.index))
    sl_mbp_7m   = df.get("slope_7m_mbp",         pd.Series(np.nan, index=df.index))

    primary_path = (
        (sl_spo2_5m  < -0.015) &
        ((sl_hr_5m   >  0.040) | (sl_etco2_5m < -0.008))
    )
    secondary_path = (
        (sl_spo2_7m < -0.010) &
        (sl_hr_7m   >  0.030) &
        (sl_mbp_7m  < -0.020)
    )
    t3_trend_activate = primary_path | secondary_path

    return {
        "t1_shock_spiral":       t1_shock_spiral,
        "t1_resp_burnout":       t1_resp_burnout,
        "t1_hypercapnic":        t1_hypercapnic,
        "t2_pulse_pressure_low": t2_pulse_pressure_low,
        "t2_widepp_highsbp":     t2_widepp_highsbp,
        "t2_resp_hemo_combo":    t2_resp_hemo_combo,
        "t3_hyper_emergency":    t3_hyper_emergency,
        "t3_stable_deceiver":    t3_stable_deceiver,
        "t3_masked_shock":       t3_masked_shock,
        "t3_occult_acidosis":    t3_occult_acidosis,
        "t3_trend_decline":      t3_trend_decline,
        "t3_trend_activate":     t3_trend_activate,
    }


# =======================
# Solar8000 → internal name mapping
# Original Solar8000 columns are KEPT in the output unchanged.
# Their values are COPIED into the internal short names used by the pipeline.
# =======================
SOLAR_TO_INTERNAL = {
    "Solar8000/PLETH_SPO2": "spo2",
    "Solar8000/HR":         "heart_rate",
    "Solar8000/RR":         "resp_rate",
    "Solar8000/ART_SBP":    "sbp",
    "Solar8000/ART_DBP":    "dbp",
    "Solar8000/ART_MBP":    "mbp",
    "Solar8000/ETCO2":      "etco2",
}


# =======================
# MAIN pipeline
# =======================
def run(input_csv, output_csv):
    df = pd.read_csv(input_csv)

    # ── Snapshot original input columns immediately ───────────────────────────
    # These are GUARANTEED to appear in the output CSV no matter what the
    # column-ordering logic below does.  Saved before any mutation of df.
    original_input_cols = list(df.columns)

    # ── Solar8000 column handling ────────────────────────────────────────────
    # If the CSV uses Solar8000/xxx names, copy their values into the internal
    # short names used throughout this pipeline.
    # The original Solar8000 columns are LEFT INTACT in df — they will appear
    # in the output CSV as metadata columns at the front (via meta_cols logic).
    solar_found = [s for s in SOLAR_TO_INTERNAL if s in df.columns]
    if solar_found:
        print(f"Solar8000 columns detected ({len(solar_found)}), mapping to internal names:")
        for solar_col, internal_col in SOLAR_TO_INTERNAL.items():
            if solar_col not in df.columns:
                continue
            if internal_col not in df.columns:
                df[internal_col] = df[solar_col]
                print(f"  {solar_col:<30s}  ->  {internal_col}  (original column kept)")
            else:
                print(f"  {solar_col:<30s}  skipped: {internal_col} already present")

    required = ["spo2", "heart_rate", "resp_rate", "sbp", "dbp", "etco2"]
    for c in required:
        if c not in df.columns:
            raise ValueError(f"Input CSV must contain column: {c}  |  columns present: {list(df.columns)}")

    patient_col = "patient_id" if "patient_id" in df.columns else None

    # ── Derived vitals ──────────────────────────────────────────────
    if "pulse_pressure" not in df.columns:
        df["pulse_pressure"] = df["sbp"] - df["dbp"]
    if "mbp" not in df.columns:
        df["mbp"] = (df["sbp"] + 2 * df["dbp"]) / 3.0

    # ── RR smoothing — strictly patient-wise to avoid cross-patient bleed ──
    if patient_col:
        df["resp_rate_smoothed"] = (
            df.groupby(patient_col)["resp_rate"]
            .transform(lambda x: x.rolling(RR_SMOOTH_WINDOW, min_periods=1).mean())
        )
    else:
        df["resp_rate_smoothed"] = df["resp_rate"].rolling(RR_SMOOTH_WINDOW, min_periods=1).mean()

    # ── Trend prev columns (patient-wise) ───────────────────────────
    for col in ["etco2", "spo2", "heart_rate"]:
        prev_col = f"{col}_prev"
        if patient_col:
            df[prev_col] = df.groupby(patient_col)[col].shift(1)
        else:
            df[prev_col] = df[col].shift(1)
        df[prev_col] = df[prev_col].fillna(df[col])

    out = df.copy()

    # ── PASS 1: z-values and severity contributions ─────────────────
    out["z_spo2"]  = out["spo2"].apply(z_spo2)
    out["z_hr"]    = out["heart_rate"].apply(z_hr)
    out["z_rr"]    = out["resp_rate_smoothed"].apply(z_rr)
    out["z_sbp"]   = out["sbp"].apply(z_sbp)
    out["z_dbp"]   = out["dbp"].apply(z_dbp)
    out["z_mbp"]   = out["mbp"].apply(z_mbp)
    out["z_etco2"] = out["etco2"].apply(z_etco2)
    out["z_pp"]    = out["pulse_pressure"].apply(z_pp)

    z_cols = ["spo2", "hr", "rr", "sbp", "dbp", "mbp", "etco2", "pp"]
    for c in z_cols:
        out[f"s_{c}"] = out[f"z_{c}"].apply(severity_from_z)
    out["severity_sum"] = out[[f"s_{c}" for c in z_cols]].sum(axis=1)

    # ── PASS 2: slopes for raw vitals (patient-wise) ─────────────────
    for vital in RAW_SLOPE_VITALS:
        if vital in out.columns:
            out = add_slopes(out, vital, patient_col=patient_col)

    # ── PASS 3: condition binary detection (uses vital slopes) ───────
    cond_binary = detect_conditions(out)
    for cond_name, series in cond_binary.items():
        out[cond_name] = series.astype(int)

    # ── PASS 4: per-row combined score ───────────────────────────────
    combined_scores = []
    severity_labels = []
    selected_1      = []
    selected_2      = []

    for idx, row in out.iterrows():
        severity_sum = float(row["severity_sum"])

        cond_infos = []
        for cond_name, defn in CONDITIONS.items():
            A_cond = max(0.0, min(1.0, compute_condition_A(row, cond_name)))
            cond_infos.append({
                "name":  cond_name,
                "tier":  defn["tier"],
                "A":     A_cond,
                "baseM": base_for_tier(defn["tier"]),
            })

        eligible = [c for c in cond_infos if c["A"] >= MIN_COND_ACTIVATION]
        if not eligible:
            eligible = [c for c in cond_infos if c["A"] > 0.0]

        # Best single condition
        best_single_core = -1.0
        best_single      = None
        for c in eligible:
            core = c["A"] * c["baseM"]
            if core > best_single_core:
                best_single_core = core
                best_single = c

        # Best pair
        best_pair_core = -1.0
        best_pair      = None
        n_elig = len(eligible)
        for i in range(n_elig):
            for j in range(i + 1, n_elig):
                ci = eligible[i]; cj = eligible[j]
                pt   = pair_target_for_tiers(ci["tier"], cj["tier"])
                core = ci["A"] * cj["A"] * pt
                if core > best_pair_core:
                    best_pair_core = core
                    best_pair = (ci, cj, pt)

        if best_pair is not None and best_pair_core > best_single_core:
            ci, cj, pair_target = best_pair
            sel_a           = ci["name"]; sel_b = cj["name"]
            A_pair          = ci["A"] * cj["A"]
            final_pair_tgt  = pair_target
        elif best_single is not None:
            sel_a           = best_single["name"]; sel_b = ""
            A_pair          = best_single["A"]
            final_pair_tgt  = best_single["baseM"]
        else:
            sel_a = ""; sel_b = ""; A_pair = 0.0; final_pair_tgt = 1.0

        M_eff       = min(1.0 + A_pair * (final_pair_tgt - 1.0), MULTIPLIER_CAP)
        final_score = float(M_eff * severity_sum)

        sev_label = (
            "Emergency" if final_score >= EMERGENCY_THRESHOLD else
            "Critical"  if final_score >= CRITICAL_THRESHOLD  else
            "Normal"
        )

        combined_scores.append(final_score)
        severity_labels.append(sev_label)
        selected_1.append(sel_a)
        selected_2.append(sel_b)

    out["combined_score"]  = combined_scores
    out["severity_label"]  = severity_labels
    out["selected_cond_1"] = selected_1
    out["selected_cond_2"] = selected_2

    # ── PASS 5: slopes for combined_score (patient-wise) ─────────────
    out = add_slopes(out, "combined_score", patient_col=patient_col)

    # ── PASS 6: rolling stats on combined_score ───────────────────────
    # roll_mean_{2m,5m,7m,15m}_combined  → rolling mean, min_periods=2
    # roll_std_{2m,5m,7m,15m}_combined   → rolling std,  min_periods=2
    # roll_min_15m_combined              → rolling min,  min_periods=1
    # roll_max_15m_combined              → rolling max,  min_periods=1
    # All NaN at patient start (no reference point) filled with 0.
    out = add_rolling_stats_combined(out, patient_col=patient_col)

    # ── PASS 7: lag-15m features ──────────────────────────────────────
    # lag_15m_{vital} for each vital in LAG_VITALS + combined_score.
    # Shift 450 rows within each patient.
    # NaN at patient start (no reference point) filled with 0.
    out = add_lag_features(out, patient_col=patient_col)

    # ── Column ordering ───────────────────────────────────────────────
    def _slope_cols_for(vital):
        return [f"slope_{w}_{vital}" for w in SLOPE_WINDOWS if f"slope_{w}_{vital}" in out.columns]

    def _roll_cols_for_combined():
        cols = []
        for w in ROLL_WINDOWS:
            for stat in ("mean", "std"):
                c = f"roll_{stat}_{w}_combined"
                if c in out.columns:
                    cols.append(c)
        for c in ("roll_min_15m_combined", "roll_max_15m_combined"):
            if c in out.columns:
                cols.append(c)
        return cols

    def _lag_cols_for(vital):
        c = f"lag_15m_{vital}"
        return [c] if c in out.columns else []

    desired = []

    # Raw vitals with their slopes interleaved
    basics = ["spo2", "heart_rate",  "resp_rate_smoothed",
              "sbp", "dbp", "mbp", "etco2", "pulse_pressure"]
    for c in basics:
        if c in out.columns:
            desired.append(c)
            if c in RAW_SLOPE_VITALS:
                desired.extend(_slope_cols_for(c))

    # z-values (no slopes)
    for c in ["z_spo2","z_hr","z_rr","z_sbp","z_dbp","z_mbp","z_etco2","z_pp"]:
        if c in out.columns: desired.append(c)

    # s-values (no slopes)
    for c in ["s_spo2","s_hr","s_rr","s_sbp","s_dbp","s_mbp","s_etco2","s_pp"]:
        if c in out.columns: desired.append(c)

    if "severity_sum" in out.columns:
        desired.append("severity_sum")

    # Per-condition binary columns — all tiers
    tier_order = ["t1_shock_spiral", "t1_resp_burnout", "t1_hypercapnic",
                  "t2_pulse_pressure_low", "t2_widepp_highsbp", "t2_resp_hemo_combo",
                  "t3_hyper_emergency", "t3_stable_deceiver", "t3_masked_shock",
                  "t3_occult_acidosis", "t3_trend_decline", "t3_trend_activate"]
    for c in tier_order:
        if c in out.columns: desired.append(c)

    # Combined score + its slopes + rolling stats
    if "combined_score" in out.columns:
        desired.append("combined_score")
        desired.extend(_slope_cols_for("combined_score"))
        desired.extend(_roll_cols_for_combined())

    # Lag features: vitals first, then combined_score
    for col in LAG_VITALS:
        desired.extend(_lag_cols_for(col))
    desired.extend(_lag_cols_for("combined_score"))

    for c in ["severity_label", "selected_cond_1", "selected_cond_2"]:
        if c in out.columns: desired.append(c)

    # ── Final column ordering ──────────────────────────────────────────────────
    # Rule 1: original input columns always appear first (in their original
    #         order), regardless of what desired/meta logic does below.
    #         This guarantees patient_id, time, dbp, mbp, heart_rate,
    #         resp_rate, sbp, spo2, etco2, pulse_pressure etc. are NEVER lost.
    # Rule 2: all engineered columns from desired are appended after.
    desired_set = set(desired)

    seen = set()
    final_cols_dedup = []

    # First: all original input columns (locked in — always present)
    for c in original_input_cols:
        if c in out.columns and c not in seen:
            final_cols_dedup.append(c)
            seen.add(c)

    # Second: engineered columns not already added by Rule 1
    for c in desired:
        if c in out.columns and c not in seen:
            final_cols_dedup.append(c)
            seen.add(c)

    # Verify nothing was lost
    missing_originals = [c for c in original_input_cols if c not in seen]
    if missing_originals:
        # Should never happen — safety net log only
        print(f"  [warn] original columns not in output (bug): {missing_originals}")

    out[final_cols_dedup].to_csv(output_csv, index=False)
    print(f"Saved: {output_csv}  ({len(out)} rows, {len(final_cols_dedup)} columns)")
    print(f"  Original input cols preserved: {[c for c in original_input_cols if c in seen]}")


# -----------------------
# CLI entry
if __name__ == "__main__":
    args = sys.argv[1:]
    if len(args) >= 2 and not args[0].startswith("-"):
        input_csv, output_csv = args[0], args[1]
    else:
        input_csv, output_csv = "/kaggle/input/datasets/arjunmahesh09999/new-beforeclean/newbefore_clean.csv", "Orginal_master_dataset.csv"
    run(input_csv, output_csv)

Saved: Orginal_master_dataset.csv  (712957 rows, 99 columns)
  Original input cols preserved: ['patient_id', 'time', 'dbp', 'mbp', 'heart_rate', 'resp_rate', 'sbp', 'spo2', 'etco2', 'pulse_pressure']


In [3]:
import pandas as pd
df=pd.read_csv("/kaggle/working/Orginal_master_dataset.csv")
df = df.sort_values(['patient_id', 'time'])


/tmp/ipykernel_55/1107150939.py:2: DtypeWarning: Columns (98) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("/kaggle/working/Orginal_master_dataset.csv")


In [4]:
# Count NaN per column
nan_counts = df.isna().sum()

# Show only columns that actually have NaN
nan_counts = nan_counts[nan_counts > 0]

print("Columns with NaN values:")
print(nan_counts)

Columns with NaN values:
slope_2m_spo2                      103
slope_5m_spo2                      103
slope_7m_spo2                      103
slope_15m_spo2                     103
slope_2m_heart_rate                103
slope_5m_heart_rate                103
slope_7m_heart_rate                103
slope_15m_heart_rate               103
slope_2m_resp_rate_smoothed        103
slope_5m_resp_rate_smoothed        103
slope_7m_resp_rate_smoothed        103
slope_15m_resp_rate_smoothed       103
slope_2m_sbp                       103
slope_5m_sbp                       103
slope_7m_sbp                       103
slope_15m_sbp                      103
slope_2m_dbp                       103
slope_5m_dbp                       103
slope_7m_dbp                       103
slope_15m_dbp                      103
slope_2m_mbp                       103
slope_5m_mbp                       103
slope_7m_mbp                       103
slope_15m_mbp                      103
slope_2m_etco2                     103


In [5]:
df.head()


,patient_id,time,dbp,mbp,heart_rate,resp_rate,sbp,spo2,etco2,pulse_pressure,...,lag_15m_resp_rate_smoothed,lag_15m_sbp,lag_15m_dbp,lag_15m_mbp,lag_15m_etco2,lag_15m_pulse_pressure,lag_15m_combined_score,severity_label,selected_cond_1,selected_cond_2
0,64,0.0,-20.0,-20.0,51.0,11.0,-20.0,97.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Emergency,t2_pulse_pressure_low,t3_masked_shock
1,64,2.0,-20.0,-20.0,51.0,11.0,-20.0,97.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Emergency,t2_pulse_pressure_low,t3_masked_shock
2,64,4.0,-20.0,-20.0,51.0,11.0,-20.0,97.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Emergency,t2_pulse_pressure_low,t3_masked_shock
3,64,6.0,-20.0,-20.0,51.0,10.0,-20.0,98.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Emergency,t2_pulse_pressure_low,t3_masked_shock
4,64,8.0,-20.0,-20.0,51.0,10.0,-20.0,98.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Emergency,t2_pulse_pressure_low,t3_masked_shock


In [6]:
df = df.fillna(0)
df = df.drop(columns=["selected_cond_1", "selected_cond_2"], errors="ignore")

In [7]:
# Count NaN per column
nan_counts = df.isna().sum()

# Show only columns that actually have NaN
nan_counts = nan_counts[nan_counts > 0]

print("Columns with NaN values:")
print(nan_counts)

Columns with NaN values:
Series([], dtype: int64)


In [8]:
df.head()

,patient_id,time,dbp,mbp,heart_rate,resp_rate,sbp,spo2,etco2,pulse_pressure,...,lag_15m_spo2,lag_15m_heart_rate,lag_15m_resp_rate_smoothed,lag_15m_sbp,lag_15m_dbp,lag_15m_mbp,lag_15m_etco2,lag_15m_pulse_pressure,lag_15m_combined_score,severity_label
0,64,0.0,-20.0,-20.0,51.0,11.0,-20.0,97.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Emergency
1,64,2.0,-20.0,-20.0,51.0,11.0,-20.0,97.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Emergency
2,64,4.0,-20.0,-20.0,51.0,11.0,-20.0,97.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Emergency
3,64,6.0,-20.0,-20.0,51.0,10.0,-20.0,98.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Emergency
4,64,8.0,-20.0,-20.0,51.0,10.0,-20.0,98.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Emergency


In [9]:
# =============================
# Severity score thresholds
# =============================
CRITICAL_THRESHOLD = 0.75
EMERGENCY_THRESHOLD = 1.4

# =============================
# FSM configuration
# =============================
WINDOW_SIZE = 15
CONFIRM_LEN = 15

EMERGENCY_UPGRADE_COUNT = 10   # >=10 Emergency in 15
NORMAL_DOWNGRADE_COUNT = 12    # >=12 Normal in 15

In [10]:
def score_to_severity(score):
    if score >= EMERGENCY_THRESHOLD:
        return 2  # Emergency
    elif score >= CRITICAL_THRESHOLD:
        return 1  # Critical
    else:
        return 0  # Normal

In [11]:
from collections import deque

def hierarchical_fsm_numeric(severity_labels):

    prev_confirmed = 0  # Start as Normal
    result = []

    window = deque(maxlen=WINDOW_SIZE)
    consec = {2: 0, 1: 0, 0: 0}

    for lab in severity_labels:

        # Update consecutive counts
        for k in consec:
            consec[k] = consec[k] + 1 if k == lab else 0

        window.append(lab)

        # ========================
        # HARD CONFIRMATION (15)
        # ========================
        if consec[2] >= CONFIRM_LEN:
            prev_confirmed = 2
            result.append(prev_confirmed)
            continue

        if consec[1] >= CONFIRM_LEN:
            prev_confirmed = 1
            result.append(prev_confirmed)
            continue

        if consec[0] >= CONFIRM_LEN:
            # Emergency → Normal must pass via Critical
            if prev_confirmed == 2:
                prev_confirmed = 1
            else:
                prev_confirmed = 0
            result.append(prev_confirmed)
            continue

        # ========================
        # WINDOW LOGIC
        # ========================
        if len(window) == WINDOW_SIZE:

            cntN = window.count(0)
            cntC = window.count(1)
            cntE = window.count(2)

            # ---- Emergency ----
            if prev_confirmed == 2:
                if cntE == 0:
                    prev_confirmed = 1
                    result.append(prev_confirmed)
                    continue

            # ---- Critical ----
            elif prev_confirmed == 1:
                if cntC == 0:
                    if cntE >= EMERGENCY_UPGRADE_COUNT:
                        prev_confirmed = 2
                        result.append(prev_confirmed)
                        continue
                    if cntN >= NORMAL_DOWNGRADE_COUNT:
                        prev_confirmed = 0
                        result.append(prev_confirmed)
                        continue

            # ---- Normal ----
            elif prev_confirmed == 0:
                if cntN == 0:
                    prev_confirmed = 1
                    result.append(prev_confirmed)
                    continue

        result.append(prev_confirmed)

    return result

In [12]:
# Drop old labels if present
for col in ["severity_label", "result_label"]:
    if col in df.columns:
        df = df.drop(columns=[col])

# Create new numeric severity
df["severity_label"] = df["combined_score"].apply(score_to_severity)

# Create result label
df["result_label"] = hierarchical_fsm_numeric(df["severity_label"].tolist())

In [13]:
import pandas as pd

# Sort properly
df = df.sort_values(['patient_id', 'time']).reset_index(drop=True)

# 900 seconds = 450 rows
shift_steps = 450

# Create future label column
df['future_label'] = (
    df.groupby('patient_id')['result_label']
    .shift(-shift_steps)
)

# Remove last 600 rows per patient
df = (
    df.groupby('patient_id', group_keys=False)
      .apply(lambda x: x.iloc[:-shift_steps] if len(x) > shift_steps else pd.DataFrame())
      .reset_index(drop=True)
)

print("Final shape:", df.shape)


/tmp/ipykernel_55/181534127.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.iloc[:-shift_steps] if len(x) > shift_steps else pd.DataFrame())


Final shape: (666607, 99)


In [14]:
df = df.groupby("patient_id").apply(lambda x: x.iloc[450:]).reset_index(drop=True)

/tmp/ipykernel_55/1285572128.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("patient_id").apply(lambda x: x.iloc[450:]).reset_index(drop=True)


In [15]:
df.to_csv('/kaggle/working/Master_DATASET.csv', index=False)

In [16]:
import pandas as pd
df=pd.read_csv("/kaggle/working/Master_DATASET.csv")
counts = (
    df.groupby("patient_id")["future_label"]
      .value_counts()
      .unstack(fill_value=0)
)

print(counts)

future_label   0.0   1.0   2.0
patient_id                    
64             183  2631  3486
84            6178   736  1036
89            7954   309   737
110           2258   198   484
117           2511   855  1434
...            ...   ...   ...
1563          1232   203   185
1580           924  2235  2901
1583            35   139  6336
1591            82   625  3193
1595          5781  1034  2125

[103 rows x 3 columns]
